[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S07_numpy_indexing_2d_datos_reales.ipynb)

# Sesión 07 · Indexing 2D y datos reales

**Módulo 2: NumPy** · ⏱️ Duración estimada: 60 minutos (más el avance del proyecto, que es trabajo aparte)

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Seleccionar celdas, filas, columnas y bloques de una matriz con `a[i, j]`, slicing, listas de índices y `np.ix_`.
2. Filtrar las filas de una tabla según la condición de una de sus columnas.
3. Cargar archivos de texto con `np.loadtxt` y `np.genfromtxt`.
4. Limpiar datos sucios: convertir valores centinela en `nan` y resumir con `np.isnan` y las funciones `nan*`.
5. Contar valores con `np.unique` y ordenar con `np.sort` y `np.argsort`.

## 📋 Qué debes saber antes
Sesiones 4 a 6: arrays, máscaras, `np.where`, arrays 2D y `axis`.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Al final está el 🧱 **Avance del proyecto** (P0 y P1): no se resuelve en este notebook, sino en tu propio repositorio.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos de práctica, escribe dos archivos CSV en la carpeta de trabajo (`ventas_tiendas.csv` y `movimientos_sucios.csv`) y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, escribe dos archivos CSV de práctica y carga los verificadores.
import copy
import csv
import hashlib
import io
import math
import statistics

import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(suppress=True)     # muestra los números sin notación científica

# ---------- Datos de práctica: ventas de tiendas ----------
nombres_tiendas = np.array(["Miraflores", "Surco", "Lince", "Barranco", "San Isidro"])
meses = np.array(["ene", "feb", "mar", "abr", "may", "jun"])
ventas_tm = rng.integers(20, 90, size=(5, 6)) * 1000              # filas: tiendas; columnas: meses

# ---------- Datos de práctica: clientes de un banco ----------
columnas_clientes = np.array(["id", "edad", "ingreso", "gasto", "canal"])   # canal: 1 app, 2 agencia, 3 cajero
_n = 15
clientes = np.column_stack([
    np.arange(101, 101 + _n),
    rng.integers(19, 70, _n),
    np.round(rng.normal(3500, 1200, _n), 2),
    np.round(rng.normal(2200, 800, _n), 2),
    rng.integers(1, 4, _n),
]).astype(float)

# ---------- Archivos para practicar la carga ----------
np.savetxt("ventas_tiendas.csv", np.column_stack([np.arange(1, 6), ventas_tm]), delimiter=",", fmt="%d",
           header="tienda,ene,feb,mar,abr,may,jun", comments="")

_filas = []
for _i in range(40):
    _campos = [str(101 + int(rng.integers(0, _n))), str(int(rng.integers(19, 70))),
               f"{rng.normal(5000, 2500):.2f}", f"{rng.normal(-150, 300):.2f}", str(int(rng.integers(1, 4)))]
    if _i in (3, 17, 29):
        _campos[1] = "-999"          # edad desconocida (valor centinela)
    if _i in (8, 22):
        _campos[2] = "-999999"       # saldo no informado (valor centinela)
    if _i in (5, 30):
        _campos[3] = ""              # monto vacío
    if _i == 12:
        _campos[1] = "NA"            # edad como texto
    if _i == 35:
        _campos[4] = ""              # canal vacío
    _filas.append(",".join(_campos))
_TEXTO_SUCIO = "id_cliente,edad,saldo,monto,canal\n" + "\n".join(_filas) + "\n"
with open("movimientos_sucios.csv", "w", encoding="utf-8") as _f:
    _f.write(_TEXTO_SUCIO)

_NOMBRES = ["nombres_tiendas", "meses", "ventas_tm", "columnas_clientes", "clientes"]
_D = copy.deepcopy({k: globals()[k] for k in _NOMBRES})
# Versiones en listas de Python: los verificadores recalculan con bucles, sin NumPy.
_L = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in _D.items()}

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _col(m, j):
    return [fila[j] for fila in m]


def _sucio():
    """El archivo sucio leído con el módulo csv, con nan en los campos vacíos o no numéricos."""
    filas = list(csv.reader(io.StringIO(_TEXTO_SUCIO)))[1:]
    salida = []
    for fila in filas:
        valores = []
        for campo in fila:
            try:
                valores.append(float(campo))
            except ValueError:
                valores.append(math.nan)
        salida.append(valores)
    return salida


def _limpio():
    m = _sucio()
    for fila in m:
        if fila[1] == -999:
            fila[1] = math.nan
        if fila[2] == -999999:
            fila[2] = math.nan
    return m


def _sin_nan(xs):
    return [x for x in xs if not math.isnan(x)]


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    v = _L["ventas_tm"]
    _esc(r, "v_surco_mar", v[1][2], "revisa la fila de Surco y la columna de marzo")
    _arr(r, "fila_lince", v[2], "debería ser la fila completa de Lince")
    _arr(r, "col_marzo", _col(v, 2), "debería ser la columna completa de marzo")
    _arr(r, "primer_trimestre", [f[:3] for f in v], "todas las tiendas, de enero a marzo")
    _arr(r, "esquina", [f[4:] for f in v[3:]], "las 2 últimas tiendas en los 2 últimos meses")
    _arr(r, "meses_pares", [[f[j] for j in (1, 3, 5)] for f in v], "todas las tiendas en febrero, abril y junio")
    _sin_cambios(r, "ventas_tm")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_forma_fila": "f0830a8d887481abce584fe85da9a7ab2b0bc6abf16f81aad7157e1a5473dca2",
        "pred_forma_fila2": "bafc77a1952eb2ef5c14f99a6aea403daade27171707882e512815dd13b9eff5",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    v = _L["ventas_tm"]
    _arr(r, "tiendas_sel", [v[0], v[3]], "las filas completas de Miraflores y Barranco, en ese orden")
    _arr(r, "meses_sel", [[f[0], f[5]] for f in v], "todas las tiendas en enero y junio")
    _arr(r, "sub", [[v[i][j] for j in (0, 2, 4)] for i in (1, 3)], "Surco y Barranco en enero, marzo y mayo (forma 2 × 3)")
    _arr(r, "por_nombre", [f for f, n in zip(v, _L["nombres_tiendas"]) if n == "Lince"],
         "filtra las filas con una condición sobre `nombres_tiendas`")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_pares": "18c986c4f9c7ccdb5e3d61ac988a24fcfa2d7d19411b4aeb05abf79806f78633",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    c = _L["clientes"]
    _arr(r, "jovenes", [f for f in c if f[1] < 30], "filas completas de los clientes con edad menor que 30")
    _arr(r, "app_alto_gasto", [f for f in c if f[4] == 1 and f[3] > 2500], "canal app y gasto mayor que 2500, las dos a la vez")
    _arr(r, "ids_ahorradores", [f[0] for f in c if f[2] - f[3] > 1500], "solo la columna id de los que ahorran más de 1500")
    cent = [f for f in c if f[1] > 100]
    _arr(r, "centenarios", cent if cent else np.empty((0, 5)), "debería ser un filtro de filas, aunque no encuentre ninguna")
    _esc(r, "n_centenarios", len(cent), "cuenta las filas de `centenarios`")
    _esc(r, "ingreso_medio_agencia", round(statistics.fmean([f[2] for f in c if f[4] == 2]), 2),
         "promedio de la columna ingreso de los clientes de agencia, con 2 decimales", tol=0.0051)
    _sin_cambios(r, "clientes")
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    v = _L["ventas_tm"]
    _arr(r, "ventas_csv", [[i + 1] + f for i, f in enumerate(v)],
         "carga el archivo con `np.loadtxt`, separador coma y saltando el encabezado")
    _arr(r, "ventas_solo_meses", v, "quítale a `ventas_csv` la primera columna (el código de tienda)")
    s = _sucio()
    _arr(r, "sucio", s, "carga el archivo con `np.genfromtxt`, separador coma y saltando el encabezado")
    _arr(r, "nan_por_columna", [len([x for x in _col(s, j) if math.isnan(x)]) for j in range(5)],
         "cuenta los `nan` de cada columna")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_loadtxt_sucio": "bd1dddfaf233665e87fb493c0364dc67523ff4525fc194616e411b16fa707f7a",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    lim = _limpio()
    _arr(r, "limpio", lim, "los centinelas -999 (edad) y -999999 (saldo) deberían quedar como `nan`, y nada más debería cambiar")
    s = globals().get("sucio")
    if isinstance(s, np.ndarray) and s.shape == (40, 5):
        if np.allclose(s, _sucio(), equal_nan=True):
            r.ok("`sucio` sigue intacto.")
        else:
            r.mal("`sucio` cambió: trabaja sobre una copia (`sucio.copy()`).")
    edades, saldos = _sin_nan(_col(lim, 1)), _sin_nan(_col(lim, 2))
    _esc(r, "edad_media", round(statistics.fmean(edades), 1), "promedio de la edad ignorando los `nan`, con 1 decimal", tol=0.051)
    _esc(r, "saldo_mediana", round(statistics.median(saldos), 2), "mediana del saldo ignorando los `nan`, con 2 decimales", tol=0.0051)
    _arr(r, "pct_faltantes", [round(len([x for x in _col(lim, j) if math.isnan(x)]) * 100 / len(lim), 1) for j in range(5)],
         "porcentaje de `nan` de cada columna de `limpio`, con 1 decimal", tol=0.051)
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_mean_nan": "7cd2a33d8476047b3755295f8b4747a3baea572e22f0e24d21505be7bc3c9844",
        "pred_nanmean": "31aaae3c5d4ba3268439fabb38e6cc6d8ce216ec1e104aedd8355912f3628b6d",
    })
    r.fin()


def check_ejercicio_6():
    r = _Revision("Ejercicio 6")
    c = _L["clientes"]
    canales = sorted(set(_col(c, 4)))
    conteos = [_col(c, 4).count(x) for x in canales]
    _arr(r, "canales_unicos", canales, "los valores distintos de la columna canal, ordenados")
    _arr(r, "conteos", conteos, "cuántas veces aparece cada canal")
    top = r.var("canal_top")
    if top is not _FALTA:
        if _es_numero(top) and float(top) in canales and conteos[canales.index(float(top))] == max(conteos):
            r.ok("`canal_top` es correcto.")
        else:
            r.mal(f"`canal_top` vale {_corto(top)}; debería ser el código del canal con más clientes (el valor, no su posición).")
    rk = r.var("ranking_tiendas")
    if rk is not _FALTA:
        totales = {n: sum(f) for n, f in zip(_L["nombres_tiendas"], _L["ventas_tm"])}
        lista = rk.tolist() if isinstance(rk, np.ndarray) else rk
        if not isinstance(rk, np.ndarray):
            r.mal(f"`ranking_tiendas` es de tipo {type(rk).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        elif sorted(lista) != sorted(totales):
            r.mal("`ranking_tiendas` debería tener los 5 nombres de tienda, cada uno una vez.")
        elif any(totales[a] < totales[b] for a, b in zip(lista, lista[1:])):
            r.mal("`ranking_tiendas` no está ordenado de la tienda que más vendió a la que menos.")
        else:
            r.ok("`ranking_tiendas` es correcto.")
    _arr(r, "ingresos_ordenados", sorted(_col(c, 2)), "la columna ingreso de menor a mayor")
    orden_ing = sorted(range(len(c)), key=lambda i: -c[i][2])
    _arr(r, "top3_ids_ingreso", [c[i][0] for i in orden_ing[:3]], "los ids de los 3 clientes con mayor ingreso, del mayor al menor")
    _sin_cambios(r, "clientes", "ventas_tm")
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    lim = _limpio()
    _arr(r, "mov", lim, "carga `movimientos_sucios.csv` y convierte los dos centinelas en `nan`")
    completas = [f for f in lim if not any(math.isnan(x) for x in f)]
    _arr(r, "filas_completas", completas, "quédate con las filas que no tienen ningún `nan`")
    _esc(r, "pct_completas", round(len(completas) * 100 / len(lim), 1), "porcentaje de filas completas, con 1 decimal", tol=0.051)
    medios = []
    for canal in (1, 2, 3):
        vals = _sin_nan([f[3] for f in lim if f[4] == canal])
        medios.append(round(statistics.fmean(vals), 2))
    _arr(r, "monto_medio_por_canal", medios,
         "el monto promedio de los canales 1, 2 y 3, en ese orden, ignorando los `nan`", tol=0.0051)
    ids = sorted(set(_col(lim, 0)))
    _arr(r, "clientes_unicos", ids, "los ids de cliente distintos, ordenados")
    _esc(r, "n_clientes", len(ids), "cuenta los clientes distintos")
    validos = [f for f in lim if not math.isnan(f[2])]
    validos.sort(key=lambda f: -f[2])
    _arr(r, "top3_saldos_ids", [f[0] for f in validos[:3]],
         "los ids de los 3 movimientos con mayor saldo, sin contar los `nan` (ojo: `argsort` los manda al final)")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    montos = sorted(_sin_nan(_col(_limpio(), 3)))
    q = statistics.quantiles(montos, n=4, method="inclusive")
    q1, q3 = q[0], q[2]
    iqr = q3 - q1
    _esc(r, "q1", q1, "el percentil 25 del monto, ignorando los `nan`", tol=1e-6)
    _esc(r, "q3", q3, "el percentil 75 del monto, ignorando los `nan`", tol=1e-6)
    atip = [x for x in _sin_nan(_col(_limpio(), 3)) if x < q1 - 1.5 * iqr or x > q3 + 1.5 * iqr]
    _arr(r, "atipicos_monto", atip, "montos fuera del rango [q1 - 1.5·IQR, q3 + 1.5·IQR], en el orden del archivo")
    r.fin()


print("✅ Setup listo. Datos generados, archivos CSV escritos y verificadores cargados.")

### 📦 Tus datos de hoy
Los valores se generan con una semilla fija, así que siempre salen iguales. `clientes` tiene una fila por cliente y las columnas de `columnas_clientes`; en la columna canal, 1 es app, 2 es agencia y 3 es cajero.

In [ ]:
print("🏪 Ventas de tiendas")
print("nombres_tiendas =", nombres_tiendas)
print("meses           =", meses)
print("ventas_tm (tiendas × meses):")
print(ventas_tm)
print()
print("🏦 Clientes de un banco")
print("columnas_clientes =", columnas_clientes)
print(clientes)
print()
print("📄 Primeras líneas de los archivos:")
for archivo in ["ventas_tiendas.csv", "movimientos_sucios.csv"]:
    with open(archivo, encoding="utf-8") as f:
        print(f"--- {archivo}")
        for _ in range(4):
            print(f.readline().rstrip())

---
## 1. Indexing y slicing en 2D

### 📘 Concepto
En una matriz, cada selección lleva **fila y columna**, separadas por coma:

| Selección | Qué devuelve |
|---|---|
| `a[i, j]` | el valor de la fila `i`, columna `j` |
| `a[i]` o `a[i, :]` | la fila `i` completa (1D) |
| `a[:, j]` | la columna `j` completa (1D) |
| `a[f1:f2, c1:c2]` | un bloque (2D) |
| `a[:, ::2]` | todas las filas, una columna sí y una no |

Detalle de forma: `a[0]` devuelve una fila en **1D**, mientras que `a[0:1]` devuelve la misma fila como matriz de **una fila** (2D). Igual que en 1D, los slices son vistas.

In [ ]:
m_ej = np.array([[10, 11, 12, 13],
                 [20, 21, 22, 23],
                 [30, 31, 32, 33]])
print(m_ej[1, 2])
print(m_ej[1], m_ej[:, 2])
print(m_ej[:2, 1:3])
print(m_ej[:, ::2])
print(m_ej[-1, -1], m_ej[0].shape, m_ej[0:1].shape)

### ✍️ Tu turno · Ejercicio 1: recortar la tabla de ventas
**Parte A.** Con `ventas_tm` (filas en el orden de `nombres_tiendas`, columnas en el orden de `meses`):
1. `v_surco_mar`: la venta de Surco en marzo.
2. `fila_lince`: todas las ventas de Lince.
3. `col_marzo`: las ventas de marzo de todas las tiendas.
4. `primer_trimestre`: todas las tiendas, de enero a marzo.
5. `esquina`: las 2 últimas tiendas en los 2 últimos meses, con índices negativos.
6. `meses_pares`: todas las tiendas en febrero, abril y junio, con slicing y paso.

**Parte B.** Predice **sin ejecutar** (una tupla, por ejemplo `(2, 5)`):

| Variable | Pregunta |
|---|---|
| `pred_forma_fila` | `ventas_tm[0].shape` |
| `pred_forma_fila2` | `ventas_tm[0:1].shape` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Surco es la fila 1 y marzo es la columna 2. Los dos puntos `:` solos significan "todas".
</details>

<details><summary>💡 Pista 2</summary>

Febrero, abril y junio son las columnas 1, 3 y 5: empieza en 1 y avanza de 2 en 2. Para `esquina` usa `-2:` en las filas y en las columnas.
</details>

---
## 2. Listas de índices y `np.ix_`

### 📘 Concepto
- `a[[0, 3]]` toma las **filas** 0 y 3, en ese orden. `a[:, [0, 5]]` toma las **columnas** 0 y 5.
- Una máscara booleana de filas también sirve: `a[nombres == "Lince"]`.
- ⚠️ `a[[1, 3], [0, 2]]` **no** da un bloque de 2 × 2: empareja los índices y devuelve los valores de las posiciones `(1, 0)` y `(3, 2)`.
- Para un bloque con filas y columnas elegidas a mano usa `a[np.ix_(filas, columnas)]`.

In [ ]:
m_ej = np.array([[10, 11, 12, 13],
                 [20, 21, 22, 23],
                 [30, 31, 32, 33]])
print(m_ej[[2, 0]])                    # filas 2 y 0
print(m_ej[:, [3, 1]])                 # columnas 3 y 1
print(m_ej[[0, 2], [1, 3]])            # posiciones (0, 1) y (2, 3)
print(m_ej[np.ix_([0, 2], [1, 3])])    # bloque 2 × 2

### ✍️ Tu turno · Ejercicio 2: elegir tiendas y meses
**Parte A.**
1. `tiendas_sel`: las filas de Miraflores y Barranco, en ese orden.
2. `meses_sel`: todas las tiendas en enero y junio.
3. `sub`: Surco y Barranco en enero, marzo y mayo (un bloque de 2 × 3), con `np.ix_`.
4. `por_nombre`: la fila de Lince, eligiéndola con una condición sobre `nombres_tiendas` (sin escribir su posición).

**Parte B.** Predice **sin ejecutar**: `pred_pares` = `ventas_tm[[1, 3], [0, 2]].shape` (una tupla).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Escribe primero las posiciones: Miraflores 0, Surco 1, Lince 2, Barranco 3; enero 0, marzo 2, mayo 4, junio 5.
</details>

<details><summary>💡 Pista 2</summary>

`np.ix_` recibe dos listas: primero las filas y luego las columnas. Para `por_nombre`, `nombres_tiendas == "Lince"` es una máscara de filas.
</details>

---
## 3. Filtrar filas según una columna

### 📘 Concepto
Es la operación más común del análisis de datos: "dame los clientes que cumplen X".
1. Toma la columna que define la condición: `tabla[:, j]`.
2. Arma la máscara: `tabla[:, j] > valor`.
3. Úsala para elegir **filas**: `tabla[mascara]` devuelve las filas completas.

Puedes combinar condiciones de varias columnas con `&` y `|`, y elegir filas y columnas a la vez: `tabla[mascara, 0]` devuelve solo la columna 0 de las filas que cumplen.

Si ninguna fila cumple, el resultado es una matriz vacía con **0 filas** y todas sus columnas, por ejemplo `(0, 5)`.

In [ ]:
tabla_ej = np.array([[1, 25, 1800.0],
                     [2, 41, 5200.0],
                     [3, 33, 2900.0]])     # columnas: id, edad, ingreso
print(tabla_ej[tabla_ej[:, 1] > 30])
print(tabla_ej[(tabla_ej[:, 1] > 30) & (tabla_ej[:, 2] < 4000)])
print(tabla_ej[tabla_ej[:, 2] > 2000, 0])          # solo los ids
print(tabla_ej[tabla_ej[:, 1] > 90].shape)          # sin coincidencias

### ✍️ Tu turno · Ejercicio 3: segmentar clientes
Con `clientes` (columnas: id, edad, ingreso, gasto, canal):
1. `jovenes`: las filas de los clientes menores de 30 años.
2. `app_alto_gasto`: las filas de los clientes de canal app (1) con gasto mayor que 2500.
3. `ids_ahorradores`: solo los **ids** de los clientes cuyo ingreso supera a su gasto en más de 1500.
4. `centenarios`: las filas de los clientes mayores de 100 años, y `n_centenarios`: cuántos son.
5. `ingreso_medio_agencia`: el ingreso promedio de los clientes de agencia (2), con 2 decimales.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Crea variables con nombre para las columnas que uses, por ejemplo `edad = clientes[:, 1]`, y arma las máscaras con ellas.
</details>

<details><summary>💡 Pista 2</summary>

Para `ids_ahorradores`, la máscara es `ingreso - gasto > 1500` y la selección es `clientes[mascara, 0]`. `n_centenarios` sale de `centenarios.shape[0]`.
</details>

---
## 4. Cargar archivos: `np.loadtxt` y `np.genfromtxt`

### 📘 Concepto
Los datos reales llegan en archivos. Un **CSV** es un archivo de texto con una fila por línea y los valores separados por comas; la primera línea suele ser el encabezado.

| Función | Cuándo usarla |
|---|---|
| `np.loadtxt(ruta, delimiter=",", skiprows=1)` | archivos **limpios**: si falta un valor o hay un texto, da error |
| `np.genfromtxt(ruta, delimiter=",", skip_header=1)` | archivos **sucios**: los valores vacíos o no numéricos se convierten en `nan` |

Las dos devuelven un array de `float` y solo leen números: una columna de texto quedaría toda en `nan`. Para archivos con texto y números mezclados usarás pandas en la sesión 9.

In [ ]:
with open("mini_ej.csv", "w") as f:
    f.write("dia,visitas,ventas\n1,120,3400\n2,,2900\n3,95,sin dato\n")

print(np.genfromtxt("mini_ej.csv", delimiter=",", skip_header=1))
# np.loadtxt("mini_ej.csv", delimiter=",", skiprows=1)   # ValueError: hay un valor vacío

### ✍️ Tu turno · Ejercicio 4: leer los dos archivos
**Parte A.**
1. `ventas_csv`: carga `ventas_tiendas.csv` con `np.loadtxt` (tiene encabezado y su primera columna es el código de tienda).
2. `ventas_solo_meses`: `ventas_csv` sin la columna del código de tienda.
3. `sucio`: carga `movimientos_sucios.csv` con `np.genfromtxt` (columnas: id_cliente, edad, saldo, monto, canal).
4. `nan_por_columna`: cuántos `nan` tiene cada columna de `sucio`. Usa `np.isnan` y `axis`.

**Parte B.** Predice **sin ejecutar**: `pred_loadtxt_sucio` es lo que devolvería `np.loadtxt("movimientos_sucios.csv", delimiter=",", skiprows=1).shape` (una tupla o `"error"`).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Copia la forma de la celda de ejemplo y cambia el nombre del archivo. Imprime la `shape` de lo que cargues.
</details>

<details><summary>💡 Pista 2</summary>

`np.isnan(sucio)` da una matriz de `bool`; súmala con `axis=0` para tener un conteo por columna.
</details>

---
## 5. Datos sucios: centinelas, `nan` y funciones `nan*`

### 📘 Concepto
Muchos sistemas marcan un dato faltante con un **valor centinela**: un número imposible como `-999`, `9999` o `-1`. Si no lo detectas, arruina los promedios. El primer paso es convertirlo en `nan`.

**Asignar con máscara** cambia solo los elementos elegidos: `a[a == -999] = np.nan`. En una matriz, limita el cambio a una columna: `m[m[:, j] == -999, j] = np.nan`. Solo funciona en arrays `float`, porque `nan` es un decimal.

Después:
- `np.isnan(a)` marca los `nan` (recuerda: `a == np.nan` no sirve).
- `np.mean`, `np.median`, `np.max`... devuelven `nan` si hay algún `nan`.
- `np.nanmean`, `np.nanmedian`, `np.nanmax`, `np.nansum`... los ignoran.

Trabaja sobre una **copia** (`.copy()`) para conservar los datos originales.

In [ ]:
edades_ej = np.array([34.0, -999.0, 27.0, 51.0, -999.0])
print(np.mean(edades_ej))                 # engañoso: incluye los -999

limpias_ej = edades_ej.copy()
limpias_ej[limpias_ej == -999] = np.nan
print(limpias_ej)
print(np.mean(limpias_ej), np.nanmean(limpias_ej))
print(np.isnan(limpias_ej).sum(), edades_ej)   # el original no cambió

### ✍️ Tu turno · Ejercicio 5: limpiar los movimientos
**Parte A.** En `sucio`, la edad desconocida se marcó con `-999` (columna 1) y el saldo no informado con `-999999` (columna 2).
1. `limpio`: una copia de `sucio` con esos centinelas convertidos en `nan`, **solo en su columna**. `sucio` no debe cambiar.
2. `edad_media`: la edad promedio de `limpio`, ignorando los `nan`, con 1 decimal.
3. `saldo_mediana`: la mediana del saldo de `limpio`, ignorando los `nan`, con 2 decimales.
4. `pct_faltantes`: el porcentaje de `nan` de cada columna de `limpio`, con 1 decimal.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_mean_nan` | `np.mean(np.array([1.0, np.nan]))` | número o `"nan"` |
| `pred_nanmean` | `np.nanmean(np.array([1.0, np.nan, 3.0]))` | número o `"nan"` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Empieza con `limpio = sucio.copy()`. Cada centinela se limpia con una asignación con máscara sobre su columna.
</details>

<details><summary>💡 Pista 2</summary>

La forma es `limpio[limpio[:, 1] == -999, 1] = np.nan`. Para `pct_faltantes`, `np.isnan(limpio).mean(axis=0)` da la proporción por columna.
</details>

---
## 6. `np.unique`, `np.sort` y `np.argsort`

### 📘 Concepto
- `np.unique(a)` devuelve los valores distintos, ordenados. Con `return_counts=True` devuelve también cuántas veces aparece cada uno: `valores, conteos = np.unique(a, return_counts=True)`.
- `np.sort(a)` devuelve una copia ordenada de menor a mayor. Para ordenar de mayor a menor, invierte: `np.sort(a)[::-1]`.
- `np.argsort(a)` devuelve las **posiciones** que ordenarían `a`. Sirve para ordenar **otro** array según este: `nombres[np.argsort(ventas)]`.

Caso borde: `np.sort` y `np.argsort` mandan los `nan` **al final**. Si luego inviertes el orden, quedan **al principio**. Filtra los `nan` antes de ordenar.

In [ ]:
canal_ej = np.array([2, 1, 1, 3, 1, 2])
print(np.unique(canal_ej, return_counts=True))

nombres_ej = np.array(["polo", "jean", "gorra"])
ventas_ej = np.array([300, 900, 150])
orden_ej = np.argsort(ventas_ej)
print(orden_ej, nombres_ej[orden_ej], nombres_ej[orden_ej[::-1]])
print(np.sort(np.array([3.0, np.nan, 1.0]))[::-1])   # el nan queda primero

### ✍️ Tu turno · Ejercicio 6: contar y ordenar
1. `canales_unicos` y `conteos`: los canales distintos de `clientes` y cuántos clientes tiene cada uno, con una sola llamada a `np.unique`.
2. `canal_top`: el **código** del canal con más clientes.
3. `ranking_tiendas`: los nombres de las tiendas ordenados de la que más vendió en total a la que menos, usando `ventas_tm`.
4. `ingresos_ordenados`: la columna ingreso de `clientes`, de menor a mayor.
5. `top3_ids_ingreso`: los ids de los 3 clientes con mayor ingreso, del mayor al menor.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_6()

<details><summary>💡 Pista 1</summary>

`canal_top` combina dos resultados: la posición del conteo máximo (`np.argmax`) sirve para indexar `canales_unicos`.
</details>

<details><summary>💡 Pista 2</summary>

Para `ranking_tiendas`: total por tienda con `axis=1`, luego `np.argsort`, invertir con `[::-1]` e indexar `nombres_tiendas`. `top3_ids_ingreso` sigue la misma idea con la columna ingreso y se queda con los 3 primeros.
</details>

---
## 🏋️ Reto final: informe de calidad del archivo de movimientos
Parte desde el archivo, como harías con datos nuevos:
1. `mov`: carga `movimientos_sucios.csv` y convierte en `nan` los dos centinelas (edad `-999` y saldo `-999999`).
2. `filas_completas`: las filas de `mov` que no tienen **ningún** `nan`, y `pct_completas`: qué porcentaje de filas representan, con 1 decimal.
3. `monto_medio_por_canal`: un array con el monto promedio de los canales 1, 2 y 3, en ese orden, ignorando los `nan` y con 2 decimales. Las filas con canal `nan` no cuentan en ningún canal.
4. `clientes_unicos`: los ids de cliente distintos, y `n_clientes`: cuántos son.
5. `top3_saldos_ids`: los ids de los 3 movimientos con mayor saldo, de mayor a menor, **sin** contar los saldos `nan`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Para las filas completas: `np.isnan(mov).any(axis=1)` marca las filas que tienen algún `nan`; niega esa máscara con `~`.
</details>

<details><summary>💡 Pista 2</summary>

Para `monto_medio_por_canal`, una comprensión de lista sobre `[1, 2, 3]` que aplique `np.nanmean` a los montos de cada canal, envuelta en `np.array` y `np.round`. Para el top 3, filtra antes las filas con saldo válido.
</details>

---
## 🚀 Nivel pro (opcional): atípicos con el rango intercuartílico
1. `q1` y `q3`: los percentiles 25 y 75 de la columna monto de `mov`, ignorando los `nan`. Investiga `np.nanpercentile`.
2. `atipicos_monto`: los montos (sin `nan`) que quedan por debajo de `q1 - 1.5 * IQR` o por encima de `q3 + 1.5 * IQR`, donde `IQR = q3 - q1`, en el orden del archivo.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P0 y P1

Hoy empieza tu proyecto de portafolio: **el sistema financiero peruano visto por sus consumidores**. Este bloque no se resuelve en este notebook, sino en tu propio repositorio. Dedícale una o dos sesiones de trabajo (P0 hoy y P1 junto con la sesión 8).

### P0 · Elegir el dataset y preparar el repositorio

**Qué hacer**
1. **Evalúa 2 o 3 fuentes candidatas.** Las de la currícula son:
   - Denuncias de consumidores en materia financiera, de seguros y pensiones (INDECOPI), en la Plataforma Nacional de Datos Abiertos (datosabiertos.gob.pe).
   - Expedientes resueltos de protección al consumidor (INDECOPI), en la misma plataforma.
   - Series de crédito, tasas y tipo de cambio del BCRP (BCRPData, estadisticas.bcrp.gob.pe).
   - Estadísticas de clientes y créditos por entidad de la SBS (sbs.gob.pe).
2. **Arma una tabla comparativa** con una fila por candidato y estas columnas: años disponibles, formato y tamaño, columnas principales, si trae una llave para cruzar fuentes (entidad, año, región), licencia y si contiene datos personales.
3. **Elige** con los criterios de la currícula: años recientes, columnas suficientes, una llave para cruzar, licencia que permita reutilizar los datos y tamaño manejable en Colab. Escribe la decisión en 3 o 4 líneas.
4. **Crea el repositorio** (nombre provisional `peru-reclamos-financieros`) con la estructura de la currícula: `README.md`, `data/`, `notebooks/`, `dashboard/` y `reports/figures/`.
5. **README borrador**: el problema en 2 o 3 líneas, las preguntas de negocio, la fuente elegida con su enlace y su licencia, y cómo reproducir el análisis.
6. **`data/README.md`**: instrucciones de descarga paso a paso (no subas los archivos pesados al repo) y un **diccionario de datos**: una fila por columna con nombre, tipo, descripción, un ejemplo y el porcentaje de faltantes, que completarás en P1.

**Por qué lo haría un analista**
Elegir la fuente es la decisión que más condiciona el proyecto: una fuente sin llave para cruzar, sin años recientes o con una licencia restrictiva te bloquea semanas después. Documentar desde el primer día la licencia, la descarga y el diccionario hace que cualquiera (incluido un reclutador) pueda reproducir tu trabajo y entender tus datos sin preguntarte.

**Cómo debe verse el resultado**
Un repositorio público con la estructura de carpetas, un README que explica en menos de un minuto qué problema abordas y con qué datos, la tabla comparativa con la decisión justificada, y `data/README.md` con la descarga y el diccionario iniciado. Revisa en la ficha de cada dataset la licencia exacta y cítala tal cual.

⚠️ **Datos personales**: si algún archivo trae información que identifique a personas (nombres, documentos, direcciones), anótalo en el diccionario; esas columnas se eliminan en la limpieza y nunca se publican.

### P1 · Primer vistazo con NumPy

**Qué hacer**
1. Descarga la fuente elegida y, si viene en Excel, guarda una copia en CSV desde tu hoja de cálculo para poder leerla con NumPy.
2. En `notebooks/01_evaluacion_fuentes.ipynb`, carga las columnas **numéricas** con `np.genfromtxt` (texto y fechas quedarán para pandas en la sesión 9).
3. Para cada columna numérica calcula: mínimo, máximo, media, mediana, cantidad de `nan` y porcentaje de faltantes.
4. Busca **valores centinela** sospechosos: números repetidos que no tienen sentido para la columna (0, -1, 999, 9999, fechas imposibles como un año 1900).
5. Para las columnas que son códigos (región, tipo de producto, entidad codificada), cuenta los valores distintos con `np.unique`.
6. Completa el diccionario de datos con lo que encontraste.

**Por qué lo haría un analista**
Antes de responder cualquier pregunta hay que saber si los datos aguantan la respuesta. Este primer vistazo detecta los problemas (faltantes, centinelas, rangos imposibles) que definirán la limpieza de P2, y evita sacar conclusiones de un promedio contaminado por un `-999`.

**Cómo debe verse el resultado**
Una tabla con una fila por columna numérica (mínimo, máximo, media, mediana y % de faltantes) y, debajo, entre 3 y 5 hallazgos escritos en lenguaje simple, por ejemplo: "la columna X tiene un 12 % de faltantes y valores 9999 que parecen centinelas; hay que convertirlos en `nan` antes de promediar". El diccionario de datos queda actualizado con esos porcentajes.

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Seleccionar una celda, una fila, una columna y un bloque de una matriz.
- [ ] Explicar la diferencia de forma entre `a[0]` y `a[0:1]`.
- [ ] Explicar por qué `a[[1, 3], [0, 2]]` no es un bloque y cuándo usar `np.ix_`.
- [ ] Filtrar las filas de una tabla según una o varias columnas.
- [ ] Elegir entre `np.loadtxt` y `np.genfromtxt` según lo limpio que esté el archivo.
- [ ] Convertir un valor centinela en `nan` solo en su columna, trabajando sobre una copia.
- [ ] Explicar la diferencia entre `np.mean` y `np.nanmean`.
- [ ] Contar valores con `np.unique` y ordenar un array según otro con `np.argsort`.
- [ ] Explicar dónde quedan los `nan` al ordenar.

**Próxima sesión (S08):** aleatorios, álgebra lineal, vectorización y el jefe final de NumPy.